# Harti Rice Price Preprocessing

In [ ]:
import pandas as pd

from harti_price_utils import preprocess_harti_prices
from paths import PREPROCESSED_PATHS, RAW_PATHS, RICE_ITEMS, save_dataset


In [ ]:
from harti_price_utils import coalesce_price_columns

df = pd.read_csv(RAW_PATHS["harti_rice"])
df = df.dropna(subset=["Item"])
df["Item"] = df["Item"].astype(str).str.strip()
df = df[(df["Item"] != "") & df["Item"].isin(RICE_ITEMS)].copy()
df["Report_Date"] = pd.to_datetime(df["Report_Date"])

df = coalesce_price_columns(df, "Pettah_Average_Current", "Pettah_Average_Previous", "Pettah_Price")
df = coalesce_price_columns(
    df, "Marandagahamula_Average_Current", "Marandagahamula_Average_Previous", "Marandagahamula_Price"
)

print("Raw rice rows kept:", len(df))
print("Pettah prices available:", df["Pettah_Price"].notna().sum())
print("Marandagahamula prices available:", df["Marandagahamula_Price"].notna().sum())


In [ ]:
pettah, pettah_long, pettah_log = preprocess_harti_prices(df, "Pettah_Price", "Pettah")
maranda, maranda_long, maranda_log = preprocess_harti_prices(df, "Marandagahamula_Price", "Marandagahamula")

save_dataset(pettah, PREPROCESSED_PATHS["harti_pettah"])
save_dataset(maranda, PREPROCESSED_PATHS["harti_marandagahamula"])
pettah_long.to_csv(PREPROCESSED_PATHS["harti_pettah_long"], index=False)
maranda_long.to_csv(PREPROCESSED_PATHS["harti_marandagahamula_long"], index=False)

unit_log = pd.concat([pettah_log, maranda_log], ignore_index=True)
unit_log.to_csv(PREPROCESSED_PATHS["harti_unit_log"], index=False)

print("Pettah wide shape:", pettah.shape, "| non-null cells:", int(pettah.notna().sum().sum()))
print("Pettah long rows kept:", len(pettah_long))
print("Marandagahamula wide shape:", maranda.shape, "| non-null cells:", int(maranda.notna().sum().sum()))
print("Marandagahamula long rows kept:", len(maranda_long))
print("Unit conversions applied:", len(unit_log))
print("Saved:", PREPROCESSED_PATHS["harti_pettah"])
